# ChemBreak20 — Cloud Notebook

Run this notebook from **Cell 1 downward**. The two targets are run sequentially and maintain separate controller state. Baseline, learning epochs, freeze, and terminal exploitation follow the locked CB20 protocol.

In [ ]:
# Cell 1 — user-visible experiment controls
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak20"
EXPERIMENT_REVISION = "CB20_ROUTE_MDP_MINI24_V1"
LIVE                = True
LIVE_PROGRESS       = True
TARGETS             = ["ChemDFM", "ChemLLM"]
RUN_BUDGET_CONTROL  = False  # publication ablation; approximately doubles query cost
print(PROJECT_ID, PROJECT_SUBDIR, EXPERIMENT_REVISION, TARGETS)

In [ ]:
# Cell 2 — clone or refresh repository
import os, subprocess, pathlib
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
REPO_ROOT = pathlib.Path(f"/content/{PROJECT_SUBDIR}_repo")
if REPO_ROOT.exists():
    subprocess.run(["git","-C",str(REPO_ROOT),"fetch","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO_ROOT),"checkout",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin",BRANCH],check=True)
else:
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_ROOT)],check=True)
PROJECT_ROOT = REPO_ROOT / PROJECT_SUBDIR
assert PROJECT_ROOT.exists(), PROJECT_ROOT
print("Project root:", PROJECT_ROOT)

In [ ]:
# Cell 3 — install pinned cloud dependencies before importing project code
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJECT_ROOT/"requirements-cloud-ml.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(PROJECT_ROOT)],check=True)
print("Dependencies installed")

In [ ]:
# Cell 4 — storage, caches, GPU
import os, pathlib
STORAGE_ROOT = pathlib.Path("/content/chembreak20_storage")
for d in [STORAGE_ROOT, STORAGE_ROOT/"cache"/"huggingface"/"hub", STORAGE_ROOT/"offload"/"ChemDFM", STORAGE_ROOT/"offload"/"ChemLLM", STORAGE_ROOT/"runs", STORAGE_ROOT/"policies"]:
    d.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(STORAGE_ROOT/"cache"/"huggingface"/"hub")
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HUB_CACHE"]
try:
    import torch
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0), "BF16:", torch.cuda.is_bf16_supported())
except Exception as e:
    print("Torch check:", e)

In [ ]:
# Cell 5 — OpenAI API key (hidden input; never written to repo/config/results)
if LIVE and not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    key = getpass("OpenAI API key (input hidden): ").strip()
    assert key, "OPENAI_API_KEY is required for GPT-5.6 Sol."
    os.environ["OPENAI_API_KEY"] = key
print("OPENAI_API_KEY:", "present" if os.environ.get("OPENAI_API_KEY") else "mock/not required")

In [ ]:
# Cell 6 — create runtime config without changing the committed config
import yaml, copy
base_config = PROJECT_ROOT / "configs" / "config.cb20.yaml"
cfg = yaml.safe_load(base_config.read_text())
cfg["run"]["dry_run"] = not LIVE
cfg["run"]["live_progress"] = LIVE_PROGRESS
cfg["run"]["experiment_revision"] = EXPERIMENT_REVISION
cfg["budget_control"]["enabled"] = RUN_BUDGET_CONTROL
runtime_config = PROJECT_ROOT / "configs" / "config.cb20.runtime.yaml"
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Runtime config:", runtime_config)
print("Attack LLM:", cfg["roles"]["attack_llm"]["model"])
print("Judge LLM:", cfg["roles"]["judge_llm"]["model"])
print("Targets:", [x["id"] for x in cfg["targets"]])

In [ ]:
# Cell 7 — selection/config preflight
from chembreak20.preflight import run_preflight
report = run_preflight(runtime_config, probe_tokenizers=LIVE, probe_roles=LIVE)
report

In [ ]:
# Cell 8 — verify exact fixed 24-task panel and immutable task-lock fields
from chembreak20.dataset import selected_tasks
tasks = selected_tasks(cfg["run"]["task_bank_path"] if os.path.isabs(cfg["run"]["task_bank_path"]) else PROJECT_ROOT/cfg["run"]["task_bank_path"],
                       cfg["run"]["mini_manifest_path"] if os.path.isabs(cfg["run"]["mini_manifest_path"]) else PROJECT_ROOT/cfg["run"]["mini_manifest_path"])
print("Tasks:", len(tasks), "Reserve:", int(tasks.is_reserve.sum()))
print("HC:", tasks.hc_id.nunique(), "HD:", tasks.hd_id.nunique(), "OT:", tasks.ot_id.nunique())
assert len(tasks)==24 and not tasks.is_reserve.any()
assert tasks.original_prompt.notna().all() and tasks.original_goal.notna().all()
print("Panel verification OK")

## Execution

For each target, CB20 performs: **Baseline → Epoch 1 → Epoch 2 → Epoch 3 → Freeze → Terminal exploitation**. The target conversation resets between learning epochs; Q-values and abstract route evidence persist. Exact candidate text is never replayed as learned memory.

In [ ]:
# Cell 9 — helper: run one target completely, then unload it
import json
from chembreak20.runner import ChemBreak20Runner
def run_target(target_id):
    runner = ChemBreak20Runner(runtime_config, target_id)
    try:
        summary = runner.run_all()
        print(json.dumps(summary, indent=2, sort_keys=True))
        return summary
    finally:
        runner.close()

In [ ]:
# Cell 10 — ChemDFM
summary_chemdfm = run_target("ChemDFM")

In [ ]:
# Cell 11 — ChemLLM (loads only after ChemDFM has been unloaded)
summary_chemllm = run_target("ChemLLM")

In [ ]:
# Cell 12 — compact cross-target summary
import pandas as pd
rows=[]
for name,s in [("ChemDFM",summary_chemdfm),("ChemLLM",summary_chemllm)]:
    rows.append({
        "target":name,
        "baseline_asr":s.get("baseline",{}).get("asr"),
        "epoch1_asr":s.get("learning_epoch_1",{}).get("asr"),
        "epoch2_asr":s.get("learning_epoch_2",{}).get("asr"),
        "epoch3_asr":s.get("learning_epoch_3",{}).get("asr"),
        "terminal_asr":s.get("terminal",{}).get("asr"),
        "terminal_asr_at_1":s.get("terminal_query_curve",{}).get("asr_at_1"),
        "terminal_asr_at_4":s.get("terminal_query_curve",{}).get("asr_at_4"),
    })
pd.DataFrame(rows)

In [ ]:
# Cell 13 — package release results for download/archive
import shutil, pathlib
run_root = STORAGE_ROOT / "runs" / EXPERIMENT_REVISION
archive = pathlib.Path(f"/content/{EXPERIMENT_REVISION}_results")
zip_path = shutil.make_archive(str(archive), "zip", root_dir=run_root)
print("Results ZIP:", zip_path)

### Publication note

The main experiment keeps ChemDFM and ChemLLM controller states separate. If you later test cross-model policy transfer, treat it as a distinct experiment rather than mixing it into these results.